# IMPTC Blind-Zone GraphML

이 Colab은 **직접 전처리 로직을 길게 작성하는 notebook**이 아니라, GitHub repo의 `scripts/`와 `src/` 코드를 실행하는 실험용 notebook입니다.

핵심 규칙:

```text
Colab = 실행 / 결과 확인
src/   = 실제 전처리 로직
scripts/ = 실행 명령어
```

기존 Colab draft의 주요 아이디어였던 blind-zone geometry, label 생성, graph 생성, visualization은 현재 repo 코드로 옮겨져 있습니다.

## 1. GitHub repo 준비

Colab 런타임이 새로 시작될 때마다 아래 셀을 실행하면 repo를 clone하거나 최신 코드로 업데이트합니다.

In [ ]:
import os

REPO_URL = 'https://github.com/jien040708/BlindSpotter.git'
REPO_DIR = '/content/BlindSpotter'

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    %cd {REPO_DIR}
    !git pull

%cd {REPO_DIR}

## 2. 패키지 설치

현재 script 실행에 필요한 최소 패키지를 설치합니다.

In [ ]:
!pip install -q -r requirements.txt
print('패키지 설치 완료')

## 3. IMPTC sample data 다운로드

데이터는 GitHub에 올리지 않습니다. 각자 Colab 또는 로컬에서 아래 script로 다운로드합니다.

In [ ]:
!bash scripts/download_imptc_sample.sh

## 4. Dataset 구조 확인

`data/sample`에 어떤 sequence, vehicle track, VRU track이 있는지 확인합니다.

In [ ]:
!python scripts/inspect_dataset.py --root data/sample

## 5. Graph 전처리 실행

IMPTC trajectory를 읽어서 frame별 graph JSON을 생성합니다.

현재 전처리에서 하는 일:

```text
vehicle / VRU trajectory 로드
reference vehicle 선택
occlusion_zone node 생성
blind_y label 생성
edge / node feature 생성
outputs/graphs 에 저장
```

In [ ]:
MAX_SEQUENCES = 4
MAX_FRAMES = 200
FRAME_STRIDE = 10

!python scripts/preprocess_sample.py   --root data/sample   --output outputs/graphs   --max-sequences {MAX_SEQUENCES}   --max-frames {MAX_FRAMES}   --frame-stride {FRAME_STRIDE}

## 6. 전처리 결과 검증

생성된 graph JSON이 올바른 구조인지 확인합니다.

검증 항목 예시:

```text
node feature dimension 일치 여부
edge_index 범위 오류 여부
edge_attr 길이 일치 여부
blind_node_indices와 blind_y 길이 일치 여부
blind node가 실제 occlusion_zone인지 여부
```

In [ ]:
!python scripts/validate_preprocessing.py --graphs outputs/graphs --write-summary

## 7. 전처리 요약 확인

생성된 graph 수, frame 수, blind-zone node 수, positive label 수를 표로 확인합니다.

In [ ]:
import json
import pandas as pd
from pathlib import Path

summary_path = Path('outputs/graphs/preprocess_summary.json')
validation_path = Path('outputs/graphs/validation_summary.json')

summary = json.loads(summary_path.read_text())
df_summary = pd.DataFrame(summary)
display(df_summary[['scene_id', 'frames', 'nodes', 'edges', 'blind_nodes', 'positive_blind_labels', 'scene_label']])

validation = json.loads(validation_path.read_text())
print('Validation errors:', validation.get('errors', 'not stored; see script output'))
print('Node types:', validation['node_types'])

## 8. Top-view 시각화 생성

전처리가 말이 되는지 눈으로 확인하기 위해 frame별 top-view PNG를 생성합니다. 주황색 영역은 polygon 기반 blind-zone 후보입니다.

In [ ]:
!python scripts/visualize_sample.py   --root data/sample   --output outputs/figures   --max-files 1   --max-frames 20

## 9. 생성된 figure 보기

몇 장만 Colab 안에서 바로 확인합니다.

In [ ]:
from IPython.display import Image, display
from pathlib import Path

figures = sorted(Path('outputs/figures').glob('*.png'))[:6]
print(f'표시할 figure 수: {len(figures)}')
for fig in figures:
    print(fig)
    display(Image(filename=str(fig)))

## 10. Python 함수로 직접 불러오기

필요하면 script만 실행하지 않고, `src/`의 함수를 직접 import해서 notebook에서 분석할 수도 있습니다.

In [ ]:
from src.imptc_dataset import load_imptc_scenes
from src.graph_builder import build_scene_graph

scenes = load_imptc_scenes(
    'data/sample',
    max_sequences=1,
    max_frames_per_sequence=120,
    frame_stride=10,
)

graph = build_scene_graph(scenes[0])
print('scene_id:', graph['scene_id'])
print('frames:', len(graph['frames']))
print('node_feature_names:', graph['node_feature_names'])
print('edge_feature_names:', graph['edge_feature_names'])

blind_nodes = sum(len(frame['blind_node_indices']) for frame in graph['frames'])
positive_blind = sum(sum(frame['blind_y']) for frame in graph['frames'])
print('blind_nodes:', blind_nodes)
print('positive_blind_labels:', positive_blind)

## 11. 코드를 수정해야 할 때

Notebook 안에서 긴 전처리 함수를 새로 만들기보다는 아래 파일을 수정합니다.

```text
IMPTC track parsing       → src/imptc_dataset.py
blind-zone node 생성      → src/graph_builder.py
blind_y label 생성        → src/label_builder.py
실행 옵션 / 저장 방식     → scripts/preprocess_sample.py
시각화                    → scripts/visualize_sample.py
전처리 검증               → scripts/validate_preprocessing.py
```

수정 후 GitHub에 push하고, Colab에서는 아래만 실행하면 최신 코드가 반영됩니다.

```python
%cd /content/BlindSpotter
!git pull
```